In [26]:
# Ignorar avisos menos relevantes
import warnings
warnings.filterwarnings("ignore")

# Bibliotecas principais de análise
import pandas as pd
import numpy as np
import time
import requests #Google Places API lib

#Precisamos desta biblioteca para podermos escolher um ficheiro local
#Se o ficheiro vier por API ou tivermos um link, é so alterar a forma de import
from google.colab import files

# 1️⃣ Faz upload do ficheiro (vai abrir uma janela para escolher no teu PC)
uploaded = files.upload()

# 2️⃣ Guarda o nome do ficheiro (Colab mostra o nome depois do upload)
filename = list(uploaded.keys())[0]

# 3️⃣ Lê o Excel, por default lê sempre a primeira tab, por isso podemos usar o "sheet_name"
# Se tivermos dados em varias tabs, devemos usar uma Dataframe(df) para cada uma das tabs
df = pd.read_excel(filename, sheet_name='Praias')
df.head()

Saving Praias.xlsx to Praias (1).xlsx


,nome_praia,latitude,longitude,Fluvial,Classe_de_Qualidade
0,Praia Fluvial da Ponte Nova,40.586049,-7.566529,1,Excelente
1,Praia Fluvial da Ponte de Soeira,41.846827,-6.936681,1,Excelente
2,Praia fluvial de Rabaçal,41.632320,-7.247094,1,Excelente
3,Praia Fluvial da Ribeira,41.585995,-6.906663,1,Excelente
4,Praia Fluvial da Fraga da Pegada,41.581312,-6.898343,1,Excelente


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1382 entries, 0 to 1381
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   nome_praia           1382 non-null   object 
 1   latitude             1382 non-null   float64
 2   longitude            1382 non-null   float64
 3   Fluvial              1382 non-null   int64  
 4   Classe_de_Qualidade  1382 non-null   object 
dtypes: float64(2), int64(1), object(2)
memory usage: 54.1+ KB


In [28]:
print("Numero de praias:", df.shape)

Numero de praias: (1382, 5)


In [29]:
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1382 entries, 0 to 1381
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   nome_praia           1382 non-null   object 
 1   latitude             1382 non-null   float64
 2   longitude            1382 non-null   float64
 3   Fluvial              1382 non-null   int64  
 4   Classe_de_Qualidade  1382 non-null   object 
dtypes: float64(2), int64(1), object(2)
memory usage: 54.1+ KB


In [30]:
def places_text_search(name, lat=None, lng=None, api_key=None):
    url = "https://places.googleapis.com/v1/places:searchText"

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": api_key,
        "X-Goog-FieldMask": (
            "places.displayName,"
            "places.location,"
            "places.rating,"
            "places.userRatingCount,"
            "places.id"
        )
    }

    body = {
        "textQuery": name
    }

    if lat is not None and lng is not None:
        body["locationBias"] = {
            "circle": {
                "center": {
                    "latitude": lat,
                    "longitude": lng
                },
                "radius": 5000.0
            }
        }

    response = requests.post(url, headers=headers, json=body)
    return response.json()

In [39]:
result = places_text_search(
    name="Praia Fluvial de Bitetos",
    lat=41.071881,
    lng=-8.259542,
    api_key=""
)

print(result)

print("Rating: " + str(result["places"][0]["rating"]))
print("userRatingCount: " + str(result["places"][0]["userRatingCount"]))
print("Id: " + result["places"][0]["id"])

{'places': [{'id': 'ChIJN-oZd4SbJA0RaHUIRwEDqy8', 'location': {'latitude': 41.0719103, 'longitude': -8.259528099999999}, 'rating': 4.4, 'userRatingCount': 1436, 'displayName': {'text': 'Praia Fluvial de Bitetos', 'languageCode': 'pt'}}]}
Rating: 4.4
userRatingCount: 1436
Id: ChIJN-oZd4SbJA0RaHUIRwEDqy8


In [40]:
df_dummy = pd.DataFrame({
    "nome_praia": pd.Series(dtype="string"),
    "latitude": pd.Series(dtype="float"),
    "longitude": pd.Series(dtype="float"),
    "Fluvial": pd.Series(dtype="int64"),
    "Classe_de_Qualidade": pd.Series(dtype="string"),
    "rating": pd.Series(dtype="float64"),
    "user_ratings_total": pd.Series(dtype="int64"),
    "place_id": pd.Series(dtype="string")
})

In [42]:
import math

sleep_time = 2
rows = []

for i in range(len(df)):
    current = df.iloc[i]

    result = places_text_search(
    name=current["nome_praia"],
    lat=current["latitude"],
    lng=current["longitude"],
    api_key=""
)
    # 1. erro da API
    if "error" in result:
        print(f"[ERROR API] linha {i} - {current['nome_praia']}")
        print(result["error"]["message"])
        continue

    # 2. sem resultados
    if "places" not in result or not result["places"]:
        print(f"[SEM PLACES] linha {i} - {current['nome_praia']}")
        print(result)
        continue

    place = result["places"][0]

    rating = place.get("rating")

    # 3. rating inválido
    if rating is None or (isinstance(rating, float) and math.isnan(rating)):
        print(f"[SEM RATING] linha {i} - {current['nome_praia']}")
        print(place)
        continue


    rows.append({
        "nome_praia": current["nome_praia"],
        "latitude": current["latitude"],
        "longitude": current["longitude"],
        "Fluvial": current["Fluvial"],
        "Classe_de_Qualidade": current["Classe_de_Qualidade"],
        "rating": result["places"][0]["rating"],
        "user_ratings_total": result["places"][0]["userRatingCount"],
        "place_id": result["places"][0]["id"],
    })
    time.sleep(sleep_time) #Sleeps for X seconds

df_dummy = pd.DataFrame(rows)

[SEM PLACES] linha 101 - Praia de Vila Praia de Âncora
{}
[SEM PLACES] linha 120 - Praia do Coral
{}
[SEM RATING] linha 130 - Ínsuas do rio Lima
{'id': 'ChIJFd5qigcJJQ0RD2OwzBXAAvQ', 'location': {'latitude': 41.8058863, 'longitude': -8.4374973}, 'displayName': {'text': 'Ínsuas do rio Lima', 'languageCode': 'fr'}}
[SEM RATING] linha 136 - Castelo de Neiva
{'id': 'ChIJU8GAaFC0JQ0R76JhWdzpfKc', 'location': {'latitude': 41.6280262, 'longitude': -8.8007806}, 'displayName': {'text': 'Castelo do Neiva', 'languageCode': 'en'}}
[SEM RATING] linha 231 - Praia Avenida da Colónia
{'id': 'ChIJXe1FL9VJJA0Rc99Fs_JjDh8', 'location': {'latitude': 41.4811002, 'longitude': -8.776209699999999}, 'displayName': {'text': 'Praia Avenida da Colónia', 'languageCode': 'de'}}
[SEM RATING] linha 234 - Praia Colónia Balnear
{'id': 'ChIJN1S8uYBJJA0RmiflCEoVp2Q', 'location': {'latitude': 41.4780071, 'longitude': -8.775790599999999}, 'displayName': {'text': 'Praia Colónia Balnear', 'languageCode': 'de'}}
[SEM PLACES] 

In [44]:
#transform df into csv
df_dummy.to_csv("praias.csv", index=False, encoding="utf-8")
#download csv
files.download("praias.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [43]:
print(df_dummy[df_dummy["place_id"].isna()])

df_dummy.shape

Empty DataFrame
Columns: [nome_praia, latitude, longitude, Fluvial, Classe_de_Qualidade, rating, user_ratings_total, place_id]
Index: []


(1306, 8)

In [8]:
df_dummy_Retries = pd.DataFrame({
    "nome_praia": pd.Series(dtype="string"),
    "latitude": pd.Series(dtype="float"),
    "longitude": pd.Series(dtype="float"),
    "Fluvial": pd.Series(dtype="int64"),
    "Classe_de_Qualidade": pd.Series(dtype="string"),
    "rating": pd.Series(dtype="float64"),
    "user_ratings_total": pd.Series(dtype="int64"),
    "place_id": pd.Series(dtype="string")
})

In [17]:
sleep_time = 5
rows = []
n= 0

for i in range(len(df_dummy)):
    current = df_dummy.iloc[i]

    if current["place_id"] == None:
        n = n + 1
        result = places_text_search(
        name=current["nome_praia"],
        lat=current["latitude"],
        lng=current["longitude"],
        api_key=""
)


    rows.append({
        "nome_praia": current["nome_praia"],
        "latitude": current["latitude"],
        "longitude": current["longitude"],
        "Fluvial": current["Fluvial"],
        "Classe_de_Qualidade": current["Classe_de_Qualidade"],
        "rating": result["places"][0]["rating"],
        "user_ratings_total": result["places"][0]["userRatingCount"],
        "place_id": result["places"][0]["id"],
    })
    time.sleep(sleep_time) #Sleeps for X seconds
print(n)
df_dummy_Retries = pd.DataFrame(rows)

1382


In [18]:
print(df_dummy_Retries[df_dummy_Retries["place_id"].isna()])

df_dummy_Retries.shape

                                        nome_praia   latitude  longitude  \
0                      Praia Fluvial da Ponte Nova  40.586049  -7.566529   
1                 Praia Fluvial da Ponte de Soeira  41.846827  -6.936681   
2                         Praia fluvial de Rabaçal  41.632320  -7.247094   
3                         Praia Fluvial da Ribeira  41.585995  -6.906663   
4                 Praia Fluvial da Fraga da Pegada  41.581312  -6.898343   
...                                            ...        ...        ...   
1377  Ponte e Parque Fluvial da Lagariça - Freigil  41.063755  -8.008263   
1378                                     Poço Azul  40.780988  -8.164787   
1379                                    Poço Negro  40.815971  -8.228202   
1380                         Praia Fluvial do Rato  40.767190  -7.740612   
1381                  Praia fluvial Ponte de Telhe  40.910493  -8.185362   

      Fluvial Classe_de_Qualidade rating user_ratings_total place_id  
0           1   

(1382, 8)

In [19]:
#transform df into csv
df_dummy_Retries.to_csv("praias.csv", index=False, encoding="utf-8")
#download csv
files.download("praias.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>